In [1]:
import os

In [2]:
%pwd

'c:\\Users\\PC Name\\Desktop\\Cheast Cancer Classification\\End-to-End-Chest-Cancer-Classification-MLflow-DVC\\research'

In [3]:
os.chdir('../')
%pwd

'c:\\Users\\PC Name\\Desktop\\Cheast Cancer Classification\\End-to-End-Chest-Cancer-Classification-MLflow-DVC'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [5]:
from src.cnnClassifier.constants import *
from src.cnnClassifier.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_roots])


    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])
    
        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion_config

In [7]:
import os
import zipfile
import gdown
import re
from src.cnnClassifier import logger
from src.cnnClassifier.utils.common import get_size

In [8]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self) -> str:
        '''
        Fetch data from the Google Drive URL, bypassing the 
        "Download anyway" virus scan warning for large files.
        '''
        try:
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            
            # Ensure target directory exists
            os.makedirs(os.path.dirname(zip_download_dir), exist_ok=True)
            logger.info(f"Downloading data from {dataset_url} into file {zip_download_dir}")

            # Robust Regex to extract Google Drive File ID from ANY format:
            # Works for: /d/FILE_ID/view, id=FILE_ID, or open?id=FILE_ID
            match = re.search(r'(r?id=|=/d/|/d/)([A-Za-z0-9_-]{33,})', dataset_url)
            
            if match:
                file_id = match.group(2)
            else:
                # Fallback if it's already just the clean ID string
                file_id = dataset_url

            logger.info(f"Extracted Google Drive File ID: {file_id}")

            # gdown handles the 'Download anyway' token natively when using the id parameter
            # setting fuzzy=True allows gdown to handle messy URLs as well
            gdown.download(id=file_id, output=zip_download_dir, quiet=False, fuzzy=True)
            
            logger.info(f"Successfully downloaded data into file {zip_download_dir}")

        except Exception as e:
            logger.error(f"Error occurred during file download: {str(e)}")
            raise e
    
    def extract_zip_file(self):
        '''
        zip_file_path: str
        Extract the zip file into the data directory
        Function return None
        '''
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [9]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-07-01 00:05:48,246: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-01 00:05:48,321: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-01 00:05:48,391: INFO: common: created directory at: artifacts]
[2026-07-01 00:05:48,392: INFO: common: created directory at: artifacts/data_ingestion]
[2026-07-01 00:05:48,396: INFO: 3183694435: Downloading data from https://drive.google.com/file/d/1Yp_28UWJa3RNZhpooiA0uJHEk9OvQfzj/view?usp=sharing into file artifacts/data_ingestion/data.zip]
[2026-07-01 00:05:48,396: INFO: 3183694435: Extracted Google Drive File ID: 1Yp_28UWJa3RNZhpooiA0uJHEk9OvQfzj]


Downloading...
From (original): https://drive.google.com/uc?id=1Yp_28UWJa3RNZhpooiA0uJHEk9OvQfzj
From (redirected): https://drive.google.com/uc?id=1Yp_28UWJa3RNZhpooiA0uJHEk9OvQfzj&confirm=t&uuid=46c9fab6-878a-497b-b163-f3458015af8a
To: c:\Users\PC Name\Desktop\Cheast Cancer Classification\End-to-End-Chest-Cancer-Classification-MLflow-DVC\artifacts\data_ingestion\data.zip
100%|██████████| 49.0M/49.0M [00:10<00:00, 4.67MB/s]

[2026-07-01 00:06:01,950: INFO: 3183694435: Successfully downloaded data into file artifacts/data_ingestion/data.zip]
